# How Diversified is an S&P 500 ETF?
## Technology Concentration and Revenue Growth in the S&P 500 at the End of 2024

**Author:** Tobias Frei · **Course:** AI Mastery Capstone, Project 2 · **Date:** 2026-11-21

**Dataset:** sp500_companies.csv from S&P 500 Stocks by Larxel, Version 1023, Kaggle, CC0 license
(https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks). 

The file describes the companies in the S&P 500 index on 2024-12-20, one row per listed ticker, with sector, market value, revenue growth and further company data. The Kaggle page does not name the source, but the columns match the data of Yahoo Finance.
It is a snapshot, so every value refers to the state of the companies at the end of 2024.

"Daily updated" means that the dataset was rebuilt every day until December 2024, not that this file contains daily data. The file has no date column. I took 2024-12-20 as the date of the snapshot because it is the last trading day in
the price files of the same version (Version 1023, published 2024-12-23).


Note: Finding the right dataset for this project was a challenge I also investigated several different Kaggle datasets, but some where to broad and deep so I decided to go with this one, since it ticked all the requirements and is suitable to answer my research questions.

### Motivation
An S&P 500 ETF is often called a safe long-term investment, because you own 500 companies. But the index is weighted by market value, so a few big technology companies get a large part of the money. In this notebook I first look at how concentrated the index is. Then I test if tech companies really grow their revenue faster than the rest, which would be one reason for their big weight. 
Disclaimer: This is not investment advice.

### Analysis Plan
I created this plan prior to the comparison of groups, so the results dont influence my hypothesis. This follows the advice from (Lusa et al., 2024) to conduct data analysis initially.

### Research Flow

- Descriptive Statistics: What share of the total market value do the sectors and the 10 biggest companies posses, comparing it to their share of all the companies.
- Test Questions: Analysis the question if the mean revenue growth of tech companies differ from the mean of all other companies.
- Outcome: Will be calculated in Revenuegrowth in %
- Groups: Will filter the companies: Technology vs. all other sectors, one row per ticker.
- Hypotheses: H0: μ_Tech = μ_Other_Sectors: They are the same: H1: μ_Tech ≠ μ_Other_Sectors (two-sided) They are significantly different testing both lower and higher contribution.
- Significance Level: I am using an alpha of 0.05
- Testing: I will be using a t-test
- Robustness Checking: I will use the bootstrap method with 95% intervals for the differences in means and in the medians with a t-test to investigate further.

All the decisions I make along the way will be documented.

### Table of Content
1. Setup
2. Ingestion & Dataset Description
3. Data Investigation and Cleaning
4. Descriptive Statistics
5. Visualization
6. Hypothesis Testing
7. Summary




---

## 1. Setup

In [17]:
# Libraries Import
import sys
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from scipy import stats


In [18]:
# Setting up configurations 
DATA = "sp500_companies.csv"
ALPHA = 0.05
SEED = 42

In [19]:
# Checking versions to avoid conflicts
print(f"Python {sys.version.split()[0]}, pandas {pd.__version__}, NumPy {np.__version__}, " f"SciPy {scipy.__version__}, Matplotlib {matplotlib.__version__}, seaborn {sns.__version__}")

Python 3.12.7, pandas 3.1.1, NumPy 2.5.5, SciPy 1.18.1, Matplotlib 3.11.2, seaborn 0.13.2


---

## 2. Ingestion & Dataset Description

In [20]:
# Loading the raw data
df_raw = pd.read_csv(DATA)
df_raw.head()

,Exchange,Symbol,Shortname,Longname,Sector,Industry,Currentprice,Marketcap,Ebitda,Revenuegrowth,City,State,Country,Fulltimeemployees,Longbusinesssummary,Weight
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309


In [21]:
df_raw.describe()

,Currentprice,Marketcap,Ebitda,Revenuegrowth,Fulltimeemployees,Weight
count,502.000000,5.020000e+02,4.730000e+02,499.000000,4.930000e+02,502.000000
mean,217.893685,1.107222e+11,7.045286e+09,0.070541,5.782761e+04,0.001992
std,489.205027,3.440074e+11,1.624213e+10,0.180247,1.395988e+05,0.006189
min,9.400000,4.664099e+09,-3.991000e+09,-0.602000,2.800000e+01,0.000084
25%,68.547500,1.933417e+10,1.625000e+09,0.002000,1.020000e+04,0.000348
50%,118.365000,3.706396e+10,2.952748e+09,0.051000,2.169000e+04,0.000667
75%,227.667500,7.830565e+10,6.020000e+09,0.109000,5.510000e+04,0.001409
max,8276.780000,3.846820e+12,1.495470e+11,1.632000,2.100000e+06,0.069209


In [22]:
df_raw.shape

(502, 16)

In [23]:
df_raw.dtypes

Exchange                   str
Symbol                     str
Shortname                  str
Longname                   str
Sector                     str
Industry                   str
Currentprice           float64
Marketcap                int64
Ebitda                 float64
Revenuegrowth          float64
City                       str
State                      str
Country                    str
Fulltimeemployees      float64
Longbusinesssummary        str
Weight                 float64
dtype: object

#### Observation
Text columns are stored as str, numbers as float64 or int64. pandas 3 uses its own data type for text, and not object.

In [24]:
# Why is Fulltimeemployees a float and not an integer?
employees = df_raw["Fulltimeemployees"]
print("Data type:", employees.dtype)


Data type: float64


In [25]:
employees.head()

0     164000.0
1      29600.0
2     228000.0
3    1551000.0
4     181269.0
Name: Fulltimeemployees, dtype: float64

In [26]:
print("Missing values:", employees.isna().sum())

# This explains why Fulltimeemployees is a float and not an integer: there are missing values.

Missing values: 9


In [27]:
# Companies with missing Fulltimeemployees values
df_raw.loc[employees.isna(), ["Symbol", "Shortname", "Sector"]]


,Symbol,Shortname,Sector
13,V,Visa Inc.,Financial Services
97,SBUX,Starbucks Corporation,Consumer Cyclical
206,DHI,"D.R. Horton, Inc.",Consumer Cyclical
257,RMD,ResMed Inc.,Healthcare
274,RJF,"Raymond James Financial, Inc.",Financial Services
384,SMCI,"Super Micro Computer, Inc.",Technology
430,FFIV,"F5, Inc.",Technology
464,SOLV,Solventum Corporation,Healthcare
501,AMTM,"Amentum Holdings, Inc.",Industrials


### Data Types Explanation
502 rows and 16 columns. Text columns are strings, numbers are float64 or int64. The column Fulltimeemployees is a float only because it has missing values. The rest does not show any anomalies compared to other datasets.


**What the columns mean** (from the Kaggle page)

| Column | Meaning |
|---|---|
| Exchange | Stock exchange (NYQ = New York Stock Exchange, NMS  = Nasdaq) where the stocks are negociated |
| Symbol | Stock Symbol (ticker) |
| Shortname, Longname | Company name |
| Sector, Industry | Sector and industry where the company operates |
| Currentprice | Share price in USD |
| Marketcap | market value of the company in USD |
| Ebitda | Earnings before interest, taxes, depreciation and amortization |
| Revenuegrowth | Revenue growth |
| City, State, Country | City, State and Country of the headquater |
| Fulltimeemployees | Number of full-time employees |
| Longbusinesssummary | Short description of the business |
| Weight | Percentage of participation on the S&P index (marketcap %) |

The file has no date column. I took 2024-12-20 as the date of the snapshot because it is the last trading day in the price files of the same version (Version 1023, published 2024-12-23).

---

## 3. Data Investigation and Cleaning
Before I elaborate on my research questions, I check missing values, duplicates, strange values categories.

In [ ]:
# Building a reusable function for profile_missing, this function will be reused after cleaning and imputing the dataset.
def profile_missing(frame: pd.DataFrame) -> pd.DataFrame:
    """Returns the number and percent of missing values for every column that has gaps."""
    counts = frame.isna().sum()
    counts = counts[counts > 0].sort_values(ascending=False)
    return pd.DataFrame({"missing": counts, "percent": (counts / len(frame) * 100).round(1)})

In [29]:
profile_missing(df_raw)

,missing,percent
Ebitda,29,5.8
State,20,4.0
Fulltimeemployees,9,1.8
Revenuegrowth,3,0.6


---

## 4. Descriptive Statistics

---

## 5. Visualization

---

## 6. Hypothesis Testing

---

## 7. Summary